In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

from dfir_copilot.engine.query_engine import QueryEngine
from dfir_copilot.detectors import available, run_detectors

engine = QueryEngine("/workspace/data/processed/three_months.parquet", timeout_s=120)
print(available())

runs = run_detectors(engine)
for run in runs:
    print(f"{run.name}: {run.status} {run.reason} | {len(run.findings)} hallazgos | {len(run.queries)} consultas")

display(pd.DataFrame([{**f.entity, "severidad": f.severity, **f.metrics}
                      for r in runs for f in r.findings]))

{'resource_breadth': 'Actores que acceden a muchos más recursos distintos que sus pares (patrón de enumeración / IDOR).'}
resource_breadth: ok  | 4 hallazgos | 4 consultas


,user_id,severidad,recursos_distintos,mediana_pares,mad,z_robusto,umbral,peticiones,ips,user_agents,pct_error,actores_comparados
0,jacintobuyer,medium,7200,5001.0,0.0,8.79,5751.1,15526,5,3,20.9,35
1,mantecoluser,medium,7183,5001.0,0.0,8.73,5751.1,15398,5,3,21.6,35
2,mateuser,medium,7177,5001.0,0.0,8.70,5751.1,15569,5,3,20.9,35
3,marcianobuyer,medium,7112,5001.0,0.0,8.44,5751.1,15468,5,3,21.4,35


In [2]:
display(pd.DataFrame(runs[0].queries)[["executed_at_utc", "status", "rows", "elapsed_ms", "sql"]])
print(runs[0].findings[0].summary)

,executed_at_utc,status,rows,elapsed_ms,sql
0,2026-10-04T23:25:08+00:00,ok,17,2,DESCRIBE logs
1,2026-10-04T23:25:08+00:00,ok,1,2,"SELECT count(""user_id"") AS ""user_id"", count(""x..."
2,2026-10-04T23:25:08+00:00,ok,1,99,"WITH per AS (SELECT ""user_id"" AS actor, count(..."
3,2026-10-04T23:25:08+00:00,ok,4,266,"SELECT ""user_id"" AS actor, count(DISTINCT ""x_i..."


jacintobuyer accedió a 7200 x_invoice_id distintos, 1.44x la mediana de sus pares (5001).


In [3]:
from dfir_copilot.detectors import correlate

runs = run_detectors(engine)
for run in runs:
    ms = sum(q["elapsed_ms"] for q in run.queries)
    print(f"{run.name}: {run.status} {run.reason} | {len(run.findings)} hallazgos | {len(run.queries)} consultas | {ms} ms")
print()
for run in runs:
    for f in run.findings:
        print(f"[{run.name}] {f.severity}: {f.summary}")

Failed to reload module 'dfir_copilot.detectors.base' from file '/workspace/src/dfir_copilot/detectors/base.py'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 455, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 407, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 455, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 367, in update_function
   

automation_clients: ok  | 4 hallazgos | 4 consultas | 221 ms
actor_ip_cluster: ok  | 1 hallazgos | 4 consultas | 396 ms
activity_ramp: ok  | 4 hallazgos | 5 consultas | 116 ms

[automation_clients] medium: mateuser: el 100% de sus peticiones (15569 de 15569) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: jacintobuyer: el 100% de sus peticiones (15526 de 15526) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: marcianobuyer: el 100% de sus peticiones (15468 de 15468) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: mantecoluser: el 100% de sus peticiones (15398 de 15398) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[actor_ip_cluster] high: 4 identidad(es) operan en exclusiva desde 5 IP(s): 12392 peticiones por IP frente a 25 típico por actor.
[activit

In [4]:
cases = correlate(runs)
display(pd.DataFrame([{"identidad": c.entity, "señales": c.signals, "severidad": c.severity,
                       "detectores": ", ".join(c.detectors)} for c in cases]))

,identidad,señales,severidad,detectores
0,jacintobuyer,3,high,"activity_ramp, actor_ip_cluster, automation_cl..."
1,mantecoluser,3,high,"activity_ramp, actor_ip_cluster, automation_cl..."
2,marcianobuyer,3,high,"activity_ramp, actor_ip_cluster, automation_cl..."
3,mateuser,3,high,"activity_ramp, actor_ip_cluster, automation_cl..."


In [6]:
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

from dfir_copilot.engine.query_engine import QueryEngine
from dfir_copilot.detectors import available, correlate, run_detectors

engine = QueryEngine("/workspace/data/processed/three_months.parquet", timeout_s=120)
print("detectores:", list(available()))

runs = run_detectors(engine)
for run in runs:
    ms = sum(q["elapsed_ms"] for q in run.queries)
    print(f"{run.name}: {run.status} {run.reason} | {len(run.findings)} hallazgos | {len(run.queries)} consultas | {ms} ms")
print()
for run in runs:
    for f in run.findings:
        print(f"[{run.name}] {f.severity}: {f.summary}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
detectores: ['automation_clients', 'actor_ip_cluster', 'activity_ramp']
automation_clients: ok  | 4 hallazgos | 4 consultas | 214 ms
actor_ip_cluster: ok  | 1 hallazgos | 4 consultas | 264 ms
activity_ramp: ok  | 4 hallazgos | 5 consultas | 66 ms

[automation_clients] medium: mateuser: el 100% de sus peticiones (15569 de 15569) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: jacintobuyer: el 100% de sus peticiones (15526 de 15526) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: marcianobuyer: el 100% de sus peticiones (15468 de 15468) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: mantecoluser: el 100% de sus peticiones (15398 de 15398) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.or

In [2]:
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

from dfir_copilot.engine.query_engine import QueryEngine
from dfir_copilot.detectors import available, correlate, run_detectors

engine = QueryEngine("/workspace/data/processed/three_months.parquet", timeout_s=120)
print("detectores:", list(available()))
assert len(available()) == 4, "Faltan detectores en el registro: reinicia el kernel (Kernel → Restart Kernel)"

runs = run_detectors(engine)
for run in runs:
    ms = sum(q["elapsed_ms"] for q in run.queries)
    print(f"{run.name}: {run.status} {run.reason} | {len(run.findings)} hallazgos | {len(run.queries)} consultas | {ms} ms")
print()
for run in runs:
    for f in run.findings:
        print(f"[{run.name}] {f.severity}: {f.summary}")

detectores: ['automation_clients', 'resource_breadth', 'actor_ip_cluster', 'activity_ramp']
automation_clients: ok  | 4 hallazgos | 4 consultas | 247 ms
resource_breadth: ok  | 4 hallazgos | 4 consultas | 424 ms
actor_ip_cluster: ok  | 1 hallazgos | 4 consultas | 387 ms
activity_ramp: ok  | 4 hallazgos | 5 consultas | 61 ms

[automation_clients] medium: mateuser: el 100% de sus peticiones (15569 de 15569) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: jacintobuyer: el 100% de sus peticiones (15526 de 15526) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: marcianobuyer: el 100% de sus peticiones (15468 de 15468) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
[automation_clients] medium: mantecoluser: el 100% de sus peticiones (15398 de 15398) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawl